# EGH444 Assessment 2 — Group workflow

Task 1 runs **both RGB baselines through one script** (`Task_1/rgb_segmentation.py`) so they share an identical protocol:

| Setting | Value (both models) |
|---|---|
| Splits | `train.txt` shuffled with seed 42 → 636 train / 159 val; `test.txt` → 654 test |
| Labels | raw IDs 1–40 → 0–39; raw 0 (unlabelled) ignored |
| Input size | 120 × 160 |
| Backbone | ImageNet ResNet50 (dilated, output stride 8), only `layer4` fine-tuned, backbone BN stats frozen |
| Optimiser | AdamW, head LR 1e-3, backbone LR 1e-4, poly decay, ≤12 epochs, early stop (patience 4), batch 2 |
| Model selection | best **val** mIoU; test evaluated only on the selected checkpoint |

The only variable is the head: **FCN** (single 3×3 conv) vs **DeepLabV3** (ASPP multi-scale context).

## Setup

In [ ]:
%pip -q install gdown

import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import gdown
import pandas as pd
from IPython.display import display

In [ ]:
# Clone the repo, or update it if it's already here.
# If the repo is ever made private again, save a GitHub token as a Colab secret named GITHUB_TOKEN.
REPO = Path("/content/EGH444-Assessment2")
BRANCH = "task1-unified-baselines"   # switch to "master" once this is merged
try:
    from google.colab import userdata
    URL = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/cal272b/EGH444-Assessment2.git"
except Exception:
    URL = "https://github.com/cal272b/EGH444-Assessment2.git"


def git(*args):
    subprocess.run(["git", *args], check=True)   # raises instead of failing silently


if REPO.exists() and not (REPO / ".git").exists():
    shutil.rmtree(REPO)                            # leftover from a failed clone
if REPO.exists():
    git("-C", REPO, "remote", "set-url", "origin", URL)
    git("-C", REPO, "fetch", "-q", "origin", BRANCH)
    git("-C", REPO, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}")
else:
    git("clone", "-q", "-b", BRANCH, URL, REPO)
print("Code at", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True).stdout)

TASK_1 = REPO / "Task_1"
SCRIPT = TASK_1 / "rgb_segmentation.py"
RUNS = TASK_1 / "runs"
sys.path.insert(0, str(TASK_1))


def run(*args):
    """Run rgb_segmentation.py and stream its output into the cell."""
    cmd = [sys.executable, str(SCRIPT), *map(str, args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"Command failed ({p.returncode})")

## Dataset

In [ ]:
DATA_ROOT = Path("/content/datasets/NYUDepthv2")

if not (DATA_ROOT / "train.txt").exists():
    zip_path = "/content/NYUDepthv2.zip"
    gdown.download(id="1nGsWNF7zU0O6RZAG-aQIs4HYnxSwP-mr", output=zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall("/content/datasets")
    Path(zip_path).unlink(missing_ok=True)
    DATA_ROOT = next(
        p.parent for p in Path("/content/datasets").rglob("train.txt")
        if (p.parent / "RGB").is_dir() and (p.parent / "Label").is_dir()
    )
print("Dataset:", DATA_ROOT)

## Task 1 — Checkpoints

Each trained model lives at `runs/<run>/<model>/best.pt`, with that run's `splits.json` two levels up
(the script refuses to evaluate a checkpoint on different sample IDs).

In [ ]:
# Back up / restore trained runs on Google Drive so a Colab reset doesn't lose them
USE_DRIVE = True
DRIVE_RUNS = Path("/content/drive/MyDrive/EGH444/runs")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)   # a stale mount gives "Transport endpoint is not connected"
    DRIVE_RUNS.mkdir(parents=True, exist_ok=True)


def restore(run_name):
    src = DRIVE_RUNS / run_name
    if USE_DRIVE and src.exists() and not (RUNS / run_name).exists():
        try:
            shutil.copytree(src, RUNS / run_name)
            print("Restored", run_name, "from Drive")
        except Exception as e:
            shutil.rmtree(RUNS / run_name, ignore_errors=True)   # don't leave a half-copied run behind
            raise RuntimeError(f"Drive copy failed ({e}). Re-run the Drive mount cell, then this cell.") from None


def backup(run_name):
    if USE_DRIVE:
        shutil.copytree(RUNS / run_name, DRIVE_RUNS / run_name, dirs_exist_ok=True)
        print("Backed up", run_name, "to Drive")

In [ ]:
# DeepLabV3 (Dylan) — existing trained weights + the original run's splits/config
DEEPLAB_RUN = "deeplab-full-corrected"
DEEPLAB_CKPT = RUNS / DEEPLAB_RUN / "deeplabv3/best.pt"
DEEPLAB_CKPT.parent.mkdir(parents=True, exist_ok=True)

if not DEEPLAB_CKPT.exists():
    gdown.download(id="1nLs0MFeoaaIBUxXlq7iwrF-pexfdW820", output=str(DEEPLAB_CKPT))
for filename in ["splits.json", "config.json"]:
    shutil.copy2(TASK_1 / "DeepLab_Results" / DEEPLAB_RUN / filename, RUNS / DEEPLAB_RUN / filename)

In [ ]:
# FCN — retrained under the SAME protocol as DeepLabV3, so only the head differs.
# The earlier template FCN run isn't comparable: 41 classes (unlabelled counted as a class),
# 240x320 input, full-backbone training, and its best epoch was picked on the test set.
FCN_RUN = "fcn-baseline"
FCN_CKPT = RUNS / FCN_RUN / "fcn/best.pt"

restore(FCN_RUN)
if not FCN_CKPT.exists():
    run("--data-root", DATA_ROOT, "--model", "fcn",
        "--output-root", RUNS, "--run-name", FCN_RUN, "--device", "auto")
    backup(FCN_RUN)
else:
    print("Using existing", FCN_CKPT)

## Task 1 — Evaluate both on val and test

In [ ]:
CHECKPOINTS = {"DeepLabV3": DEEPLAB_CKPT, "FCN": FCN_CKPT}
INCLUDE_TEST = True   # both checkpoints were selected on val only, so reporting test is legitimate

EVAL = {}
for name, ckpt in CHECKPOINTS.items():
    eval_name = f"{name.lower()}-eval"
    shutil.rmtree(RUNS / eval_name, ignore_errors=True)   # the script won't overwrite a run folder
    args = ["--data-root", DATA_ROOT, "--evaluate-checkpoint", ckpt,
            "--output-root", RUNS, "--run-name", eval_name, "--device", "auto"]
    if not INCLUDE_TEST:
        args.append("--validation-only")
    run(*args)
    EVAL[name] = RUNS / eval_name

## Task 1 — Side-by-side comparison

In [ ]:
import importlib
importlib.invalidate_caches()   # Task_1 may not have existed when sys.path was first scanned
from compare_runs import compare

COMPARISON = TASK_1 / "comparison"
summary = compare(EVAL, out_dir=COMPARISON)
display(pd.DataFrame(summary).set_index(["split", "model"]))

In [ ]:
# Classes where the two heads differ most on val (positive = DeepLabV3 better)
classwise = pd.read_csv(COMPARISON / "classwise_iou_val.csv")
diff_col = next(c for c in classwise if c.startswith("diff_"))
display(classwise.reindex(classwise[diff_col].abs().sort_values(ascending=False).index).head(10))

In [ ]:
# Weakest val images per model — starting point for the Task 2 failure analysis
for name, folder in EVAL.items():
    per_image = pd.read_csv(folder / "val/per_image.csv").sort_values("mIoU")
    print(name)
    display(per_image.head(5))

## Task 2 — RGB improvements

Each experiment changes **one thing** against its own baseline (same splits, seed, recipe):

| Run | Baseline | Change | Motivation |
|---|---|---|---|
| A | DeepLabV3 | ASPP rates 12/24/36 → 3/6/9 | At 120×160 the feature map is 15×20, so rates 24/36 fall entirely outside it and ASPP degenerates to 1×1 convs (Chen et al. 2017, §3.3). Task 1 showed no gain over FCN. |
| B | FCN | Cross-entropy → median-frequency weighted CE | Rare classes (bag, box, floor mat) near 0% IoU; class imbalance (Eigen & Fergus 2015). |
| C | FCN | Input 120×160 → 240×320 | A and B gave no gain, and training loss keeps falling while val mIoU plateaus: the limit is spatial detail (15×20 feature map), not the head or loss. Both scored on **240×320 labels** so they're judged against the same masks. |
| Seeds | FCN | Training seeds 1 and 2 (same splits) | Measures run-to-run noise, so A/B/C differences can be judged against it. |

In [ ]:
def evaluate_ckpt(label, ckpt, eval_size=None):
    """Evaluate a checkpoint on val+test (optionally at a fixed label resolution) and register it in EVAL."""
    eval_name = label.lower().replace(" ", "-").replace("@", "at") + "-eval"
    shutil.rmtree(RUNS / eval_name, ignore_errors=True)
    args = ["--data-root", DATA_ROOT, "--evaluate-checkpoint", ckpt, "--output-root", RUNS, "--run-name", eval_name]
    if eval_size:
        args += ["--eval-size", *eval_size]
    run(*args)
    EVAL[label] = RUNS / eval_name


def train(run_name, *flags):
    """Train a run unless it's already trained (locally or on Drive). Returns its checkpoint."""
    restore(run_name)
    model = flags[flags.index("--model") + 1]
    ckpt = RUNS / run_name / model / "best.pt"
    if not ckpt.exists():
        run("--data-root", DATA_ROOT, "--output-root", RUNS, "--run-name", run_name, *flags)
        backup(run_name)
    return ckpt


def show_comparison(baseline, label):
    out = TASK_1 / f"comparison_{label.lower().replace(' ', '-')}"
    summary = compare({baseline: EVAL[baseline], label: EVAL[label]}, out_dir=out)
    display(pd.DataFrame(summary).set_index(["split", "model"]))
    return out

In [ ]:
# A — DeepLabV3 with ASPP rates sized to the 15x20 feature map
evaluate_ckpt("DeepLabV3 ASPP 3-6-9", train("deeplab-aspp369", "--model", "deeplabv3", "--aspp-rates", 3, 6, 9))
CMP_A = show_comparison("DeepLabV3", "DeepLabV3 ASPP 3-6-9")

In [ ]:
# B — FCN with median-frequency class-weighted cross-entropy
evaluate_ckpt("FCN weighted-CE", train("fcn-weightedce", "--model", "fcn", "--loss", "weighted-ce"))
CMP_B = show_comparison("FCN", "FCN weighted-CE")

In [ ]:
# C — FCN at 240x320 input (~4x compute, ~20 min to train). Both scored on 240x320 labels.
evaluate_ckpt("FCN @240 labels", FCN_CKPT, eval_size=(240, 320))
evaluate_ckpt("FCN 240x320", train("fcn-240x320", "--model", "fcn", "--size", 240, 320), eval_size=(240, 320))
CMP_C = show_comparison("FCN @240 labels", "FCN 240x320")

In [ ]:
# Seeds — FCN baseline retrained with two more training seeds on the SAME splits (~6 min each)
seed_rows = []
for seed in [42, 1, 2]:
    label = "FCN" if seed == 42 else f"FCN seed {seed}"
    if seed != 42:
        evaluate_ckpt(label, train(f"fcn-seed{seed}", "--model", "fcn", "--seed", seed, "--split-seed", 42))
    for split in ["val", "test"]:
        m = json.loads((EVAL[label] / split / "metrics.json").read_text())
        seed_rows.append({"seed": seed, "split": split, "mIoU_%": 100 * m["mIoU"], "PixelAcc_%": 100 * m["PixelAcc"]})
seeds = pd.DataFrame(seed_rows)
display(seeds.groupby("split")[["mIoU_%", "PixelAcc_%"]].agg(["mean", "std", "min", "max"]).round(2))
print("Rule of thumb: a difference smaller than ~2 std of mIoU is within run-to-run noise.")

In [ ]:
# Training cost: time per epoch, best epoch, peak GPU memory
rows = []
for run_name in [FCN_RUN, "deeplab-aspp369", "fcn-weightedce", "fcn-240x320", "fcn-seed1", "fcn-seed2"]:
    for history in (RUNS / run_name).glob("*/training.csv"):
        h = pd.read_csv(history)
        rows.append({"run": run_name, "epochs_run": len(h),
                     "best_epoch": int(h.loc[h.val_mIoU.idxmax(), "epoch"]),
                     "sec_per_epoch": round(h.epoch_seconds.mean(), 1),
                     "peak_GPU_MB": round(h.peak_gpu_MB.max()) if "peak_gpu_MB" in h and h.peak_gpu_MB.notna().any() else "n/a (older run)"})
display(pd.DataFrame(rows))

In [ ]:
# Which classes each change helped or hurt most on test (positive = baseline better)
for out in [CMP_A, CMP_B, CMP_C]:
    t = pd.read_csv(out / "classwise_iou_test.csv")
    d = next(c for c in t if c.startswith("diff_"))
    print(out.name)
    display(t.reindex(t[d].abs().sort_values(ascending=False).index).head(8))

## Task 2 — RGB failure analysis (motivates Task 3)

Quantitative: where errors concentrate (boundaries vs interiors), which classes get confused, and which scene types are hardest.
Qualitative: the worst test images. Look for confusions that **geometry** could resolve (e.g. floor vs floor mat, wall vs door/whiteboard,
table vs desk) — those are the cases depth should help in Task 3.

In [ ]:
from failure_analysis import top_confusions, scene_breakdown, boundary_summary, worst_images, show_examples

print("Accuracy near class boundaries vs object interiors (test)")
display(boundary_summary({k: EVAL[k] for k in ["DeepLabV3", "FCN", "FCN @240 labels", "FCN 240x320"]}))

In [ ]:
print("Most frequent confusions — FCN baseline (test)")
display(top_confusions(EVAL["FCN"], "test"))

In [ ]:
print("Per-image mIoU by scene brightness and clutter (test, both scored on 240x320 labels)")
display(scene_breakdown({"FCN 120x160": EVAL["FCN @240 labels"], "FCN 240x320": EVAL["FCN 240x320"]}))

In [ ]:
# Worst test images for the FCN baseline, then all models side by side on them
worst = worst_images(EVAL["FCN"], "test", n=6)
display(worst)
show_examples({k: EVAL[k] for k in ["DeepLabV3", "FCN", "FCN weighted-CE"]},
              worst["id"].astype(str).tolist(), DATA_ROOT, "test", title="Worst FCN test images (120x160)")